<a href="https://colab.research.google.com/github/hmmnyamminji/DL/blob/main/day15_practice2_%EC%9E%84%EB%B2%A0%EB%94%A9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
import torch
import torch.nn as nn
import torch.nn.functional as F # 신경망에서 쓰는 함수를 모아둔 모듈
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
import matplotlib.pyplot as plt

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"

In [13]:
data = [
    ("이 영화 정말 재미있다", 1), ("정말 최고의 영화", 1),
    ("연기가 훌륭하다", 1), ("스토리가 재미있다", 1),
    ("배우가 최고다", 1), ("정말 훌륭하다", 1),
    ("음악이 아름답다", 1), ("연출이 훌륭하다", 1),
    ("최고다 재미있다", 1), ("아름답다 최고다", 1),
    ("이 영화 너무 지루하다", 0), ("최악이다 정말 지루하다", 0),
    ("연기가 어색하다", 0), ("스토리가 지루하다", 0),
    ("배우가 최악이다", 0), ("정말 어색하다", 0),
    ("음악이 끔찍하다", 0), ("연출이 최악이다", 0),
    ("끔찍하다 지루하다", 0), ("어색하다 끔찍하다", 0),
]

In [14]:
# 1. 단어 사전 만들기
vocab = {"<pad>":0, "<unk>":1}
for s, _ in data:
  for tck in s.split():
    vocab.setdefault(tck, len(vocab)) # 없을 때만 추가

print(f"사전 크기: {len(vocab)}")
print(vocab)

사전 크기: 20
{'<pad>': 0, '<unk>': 1, '이': 2, '영화': 3, '정말': 4, '재미있다': 5, '최고의': 6, '연기가': 7, '훌륭하다': 8, '스토리가': 9, '배우가': 10, '최고다': 11, '음악이': 12, '아름답다': 13, '연출이': 14, '너무': 15, '지루하다': 16, '최악이다': 17, '어색하다': 18, '끔찍하다': 19}


In [15]:
# 2. 정수 인코딩 - 각 문장을 정수 텐서로
def encode(s, max_len=4):
  ids = [vocab.get(t, 1) for t in s.split()][:max_len] # 문장을 번호 리스트로 변환 후 앞 4개만
  return ids + [0] * (max_len - len(ids))

X = torch.tensor([encode(s) for s, _ in data])
y = torch.tensor([lab for _, lab in data], dtype=torch.float32).reshape(-1,1) # 2차원 배열로 변환
print(X)
print(y)

tensor([[ 2,  3,  4,  5],
        [ 4,  6,  3,  0],
        [ 7,  8,  0,  0],
        [ 9,  5,  0,  0],
        [10, 11,  0,  0],
        [ 4,  8,  0,  0],
        [12, 13,  0,  0],
        [14,  8,  0,  0],
        [11,  5,  0,  0],
        [13, 11,  0,  0],
        [ 2,  3, 15, 16],
        [17,  4, 16,  0],
        [ 7, 18,  0,  0],
        [ 9, 16,  0,  0],
        [10, 17,  0,  0],
        [ 4, 18,  0,  0],
        [12, 19,  0,  0],
        [14, 17,  0,  0],
        [19, 16,  0,  0],
        [18, 19,  0,  0]])
tensor([[1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.]])


In [22]:
# 셀 2. 임베딩 (nn.Embedding)
EMB_DIM = 8 # 단어 하나 = 숫자 8개
emb = nn.Embedding(len(vocab), EMB_DIM, padding_idx=0) # 단어사전 크기, 벡터 차원 수, 0벡터 (학습 제외)
print(emb)
print(emb.weight)
# 임베딩: 단어 하나를 벡터로 만든다. 단어 하나(번호)를 정해진 길이의 숫자 목록으로 바꾼다. 처음엔 이 8개 숫자가 그냥 랜덤이라 아무 의미가 없다.
# 학습하면서 이 숫자들도 함께 갱신되고, 학습이 끝나면 비슷한 뜻의 단어는 벡터가 서로 가까워지고, 반대 뜻은 멀어진다.
# 이 상대적 위치 관계가 곧 의미다. 임베딩 학습은 이 "의미의 지도"에서 각 단어의 자리를 잡아주는 과정이고, 비슷한 단어끼리 가까이 모이도록 배치

Embedding(20, 8, padding_idx=0)
Parameter containing:
tensor([[ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000],
        [-0.4503, -0.0730, -0.5480, -1.1426, -0.4488, -0.0305,  0.3830, -0.0448],
        [ 1.1799, -0.3314,  0.6495,  0.0950, -0.7526, -0.6472, -1.2823,  1.9653],
        [-0.9638, -2.5668,  0.7096,  0.8198,  0.6214,  0.4232, -0.3389,  0.5180],
        [-1.3638,  0.1930, -0.6103,  0.1632,  1.5102,  0.2123, -0.7252, -0.9528],
        [ 0.5217, -0.4639,  0.1824, -0.3867, -1.7907,  0.0933, -1.9153, -0.6422],
        [ 1.3439, -1.2922,  0.7662,  0.6454,  0.3533, -2.6475, -1.4575, -0.9712],
        [ 0.2540, -0.1791,  1.1993, -0.4292,  1.0103,  0.6110,  1.2208, -0.6076],
        [-1.7376, -0.1254, -1.3658,  1.1117, -0.6228, -0.7892, -0.1678,  1.6433],
        [ 2.0071, -1.2531,  1.1189,  1.7733, -2.0717, -0.4125, -0.9770, -0.0336],
        [ 1.8595,  2.6221,  0.3691,  0.3803,  0.1990, -0.2361,  0.3034, -0.4501],
        [ 0.4739,  0.6503,  1.1662,  0.0169,

In [24]:
idx = torch.tensor(vocab["영화"]) # 단어사전 번호 3
print("\nembedding(영화): ", [round(v, 2) for v in emb(idx).tolist()])
print("weight[영화 행]: ", [round(v, 2) for v in emb.weight[idx].tolist()])
assert torch.equal(emb(idx), emb.weight[idx])
print("완전 일치: 임베딩 = 표에서 해당 행을 꺼내는 것")
# 임베딩 = (사전크기20 x 8차원벡터) 표 하나
# 최신 LLM 임베딩 = 단어사전 20만개 x 1만 차원의 벡터



embedding(영화):  [-0.96, -2.57, 0.71, 0.82, 0.62, 0.42, -0.34, 0.52]
weight[영화 행]:  [-0.96, -2.57, 0.71, 0.82, 0.62, 0.42, -0.34, 0.52]
완전 일치: 임베딩 = 표에서 해당 행을 꺼내는 것


In [27]:
# 셀 3. 학습 전 스냅샷 - 지금 유사도는 '무의미한 랜덤'
def sim(w1, w2, table):
  v1, v2 = table[vocab[w1]], table[vocab[w2]] # 두 단어의 벡터를 임베딩(표)에서 꺼냄
  return F.cosine_similarity(v1, v2, dim=0).item() # 코사인 유사도: 두 벡터의 '방향'이 비슷한가
before = emb.weight.detach().clone()
pairs = [("재미있다", "최고다"), ("재미있다", "지루하다"), ("훌륭하다", "최악이다")]
print("\n[학습 전 - 랜덤 초기값]")
for a,b in pairs:
  print(f" {a} ↔ {b}: {sim(a,b, before):+.2f}")


[학습 전 - 랜덤 초기값]
 재미있다 ↔ 최고다: -0.19
 재미있다 ↔ 지루하다: -0.38
 훌륭하다 ↔ 최악이다: +0.03


In [28]:
# 셀 4. 감성 분류로 임베딩 학습 - 의미가 스며든다

class TinySentiment(nn.Module):
  def __init__(self):
    super().__init__()
    self.emb = emb              # 위에서 만든 임베딩(표) 그대로 사용
    self.fc = nn.Linear(EMB_DIM, 1) # 문장벡터(8차원) → 점수 1개 (긍/부정 판단부)

  def forward(self, x):
    vecs = self.emb(x)          # 1. 문장이 들어오면 각 단어를 벡터로 바꾸고
    sent = vecs.mean(dim=1)     # 2. dim=1(단어축) 그 벡터들을 평균 내서 문장 하나를 대표하는 벡터로 만들고
    return torch.sigmoid(self.fc(sent)) # 3. 그걸로 긍정 확률(0-1)을 뱉는다.


model = TinySentiment()
loss_fn = nn.BCELoss()
opt = torch.optim.Adam(model.parameters(), lr=0.05)

for epoch in range(300):
  loss = loss_fn(model(X), y)
  opt.zero_grad()
  loss.backward()
  opt.step()
print(f"\n 학습 완료 (loss {loss.item():.4f})")

after = emb.weight.detach() # 학습 후 임베딩
print("\n [학습 후 - 감성 과제를 풀며 생긴 의미]")
for a, b in pairs:
  print(f" {a} ↔ {b}: 학습 전 {sim(a, b, before):+.2f} → 학습 후 {sim(a, b, after):+.2f}")


 학습 완료 (loss 0.0002)

 [학습 후 - 감성 과제를 풀며 생긴 의미]
 재미있다 ↔ 최고다: 학습 전 -0.19 → 학습 후 +0.85
 재미있다 ↔ 지루하다: 학습 전 -0.38 → 학습 후 -0.80
 훌륭하다 ↔ 최악이다: 학습 전 +0.03 → 학습 후 -0.81
